# LLM-as-Judge Evaluation: Retail Chatbot Scenario

This notebook implements and runs the LLM-as-judge evaluation designed in `evaluation_design.md`, for the retail customer service chatbot scenario described in `README.md`.


## 1. Setup and environment

In [ ]:
# %pip install openai python-dotenv -q

In [2]:
import json
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # loads OPENAI_API_KEY from .env

client = OpenAI()

GENERATOR_MODEL = "gpt-4o-mini"
JUDGE_MODEL = "gpt-4o-mini"

# Pricing per 1M tokens, gpt-4o-mini, as of this lab. Update if pricing changes.
PRICE_PER_1M_INPUT = 0.15
PRICE_PER_1M_OUTPUT = 0.60

print("Client ready. Generator model:", GENERATOR_MODEL, "| Judge model:", JUDGE_MODEL)

Client ready. Generator model: gpt-4o-mini | Judge model: gpt-4o-mini


## 2. Judge function

This implements the judge prompt from `evaluation_design.md`, built for Prompt #3 (angry customer complaint). It scores a chatbot response on 3 criteria: acknowledgment, concrete resolution, and tone.

In [3]:
JUDGE_SYSTEM_PROMPT = """You are an evaluation judge for a retail customer service chatbot.

Task Description:
The model was given an angry customer complaint about a broken product,
and asked to respond as the retailer's customer service chatbot.

Evaluation Criteria:
1. Acknowledgment: Does the response specifically acknowledge the customer's
   complaint (the broken product, their frustration), rather than a generic
   "sorry for the inconvenience"?
2. Concrete resolution: Does the response offer one clear, policy-consistent
   next step (refund, replacement, or escalation to a human), instead of
   vague reassurance with no action?
3. Tone: Is the language calm and professional, free of defensiveness,
   over-apologizing, or blaming the customer?

Reasoning Steps:
Step 1: Identify whether the response names the specific problem the
customer raised, or only gives a generic apology.
Step 2: Identify whether the response offers exactly one clear, actionable
next step for the customer to take or expect.
Step 3: Read the tone of the language for defensiveness, dismissiveness, or
excessive apology that does not address the issue.

Output only a JSON object in this exact format:
{
  "score": <integer 1-5>,
  "reasoning": "<short explanation of the score>",
  "criteria_met": {
    "acknowledgment": <true/false>,
    "concrete_resolution": <true/false>,
    "tone": <true/false>
  }
}
"""


def run_judge(prompt: str, response: str) -> dict:
    """Scores a chatbot response to a customer prompt. Returns score, reasoning,
    criteria_met, time_seconds, and token usage."""
    start = time.time()
    completion = client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Customer complaint:\n{prompt}\n\nChatbot response to evaluate:\n{response}",
            },
        ],
    )
    elapsed = time.time() - start

    try:
        judged = json.loads(completion.choices[0].message.content)
    except json.JSONDecodeError:
        judged = {"score": None, "reasoning": "JSON parse error", "criteria_met": {}}

    usage = completion.usage
    return {
        "score": judged.get("score"),
        "reasoning": judged.get("reasoning"),
        "criteria_met": judged.get("criteria_met", {}),
        "time_seconds": elapsed,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
    }


# Quick smoke test
test_result = run_judge(
    prompt="This is the worst product I've ever bought. It broke after two days and you are basically scamming your customers.",
    response="I'm sorry you're upset. Have a nice day.",
)
test_result

{'score': 1,
 'reasoning': "The response fails to acknowledge the specific complaint about the broken product and the customer's frustration. It does not offer any resolution or next steps, and the tone is dismissive rather than professional.",
 'criteria_met': {'acknowledgment': False,
  'concrete_resolution': False,
  'tone': False},
 'time_seconds': 3.5366129875183105,
 'input_tokens': 358,
 'output_tokens': 83}

## 3. Test dataset

The judge above is built for complaint-response quality (Prompt #3 in `evaluation_design.md`), so the test set below uses 5 complaint variations rather than mixing in the other 4 prompt types, which need different judges. This also matches the calibration approach from `evaluation_design.md`: a mix of clearly good, clearly bad, and borderline responses, so I can sanity-check whether the judge's scores make sense.

Two responses are generated live from the model, and three are fixed examples written to control for a known good, bad, or borderline quality, since I need known-quality references to sanity-check the judge, not just realistic ones.

In [4]:
GENERATOR_SYSTEM_PROMPT = """You are a customer service chatbot for an online retailer.
Be helpful, calm, and professional. Acknowledge the customer's specific issue
and offer one clear next step (refund, replacement, or escalation to a human)."""


def generate_response(customer_message: str) -> str:
    completion = client.chat.completions.create(
        model=GENERATOR_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": customer_message},
        ],
    )
    return completion.choices[0].message.content.strip()


# Two live prompts, responses generated by the model itself
live_prompts = [
    "This is the worst product I've ever bought. It broke after two days and you are basically scamming your customers.",
    "Great, ANOTHER broken item from you guys. Third time this month. Are you even trying?",
]

test_cases = []
for i, prompt in enumerate(live_prompts, start=1):
    test_cases.append(
        {
            "id": f"live_{i}",
            "prompt": prompt,
            "response": generate_response(prompt),
            "expected_quality": "unknown (live generation, judge score is the real test)",
        }
    )

# Three fixed examples with a known, controlled quality, used as calibration references
fixed_cases = [
    {
        "id": "fixed_good",
        "prompt": "I ordered a blender and it arrived cracked. This is unacceptable.",
        "response": (
            "I'm sorry your blender arrived cracked, that's not the experience we want for you. "
            "I can send a free replacement right away, or issue a full refund if you'd prefer. "
            "Which would you like me to set up?"
        ),
        "expected_quality": "good",
    },
    {
        "id": "fixed_bad_generic",
        "prompt": "I ordered a blender and it arrived cracked. This is unacceptable.",
        "response": "I'm sorry for the inconvenience. Thank you for your patience.",
        "expected_quality": "bad",
    },
    {
        "id": "fixed_borderline_vague",
        "prompt": "I ordered a blender and it arrived cracked. This is unacceptable.",
        "response": "I understand your blender arrived cracked, that's frustrating. We'll look into it and get back to you.",
        "expected_quality": "borderline",
    },
]

test_cases.extend(fixed_cases)

for case in test_cases:
    print(case["id"], "->", case["response"][:80], "...")

live_1 -> I’m sorry to hear that you had such a negative experience with your purchase. I  ...
live_2 -> I’m really sorry to hear that you’ve received another broken item. I understand  ...
fixed_good -> I'm sorry your blender arrived cracked, that's not the experience we want for yo ...
fixed_bad_generic -> I'm sorry for the inconvenience. Thank you for your patience. ...
fixed_borderline_vague -> I understand your blender arrived cracked, that's frustrating. We'll look into i ...


## 4. Run evaluation and collect metrics

Run the judge on all 5 test cases, and record score, reasoning, criteria met, time, tokens, and cost for each. Then compute aggregate statistics and save everything to `evaluation_results.json`.

In [6]:
def estimate_cost(input_tokens: int, output_tokens: int) -> float:
    return (input_tokens / 1_000_000) * PRICE_PER_1M_INPUT + (
        output_tokens / 1_000_000
    ) * PRICE_PER_1M_OUTPUT


run_start = time.time()
individual_results = []

for case in test_cases:
    judged = run_judge(prompt=case["prompt"], response=case["response"])
    cost = estimate_cost(judged["input_tokens"], judged["output_tokens"])
    individual_results.append(
        {
            "id": case["id"],
            "prompt": case["prompt"],
            "response": case["response"],
            "expected_quality": case["expected_quality"],
            "score": judged["score"],
            "reasoning": judged["reasoning"],
            "criteria_met": judged["criteria_met"],
            "time_seconds": round(judged["time_seconds"], 3),
            "input_tokens": judged["input_tokens"],
            "output_tokens": judged["output_tokens"],
            "estimated_cost_usd": round(cost, 6),
        }
    )

total_time = time.time() - run_start

scores = [r["score"] for r in individual_results if r["score"] is not None]
aggregate = {
    "num_cases": len(individual_results),
    "average_score": round(sum(scores) / len(scores), 2) if scores else None,
    "min_score": min(scores) if scores else None,
    "max_score": max(scores) if scores else None,
    "total_time_seconds": round(total_time, 3),
    "total_input_tokens": sum(r["input_tokens"] for r in individual_results),
    "total_output_tokens": sum(r["output_tokens"] for r in individual_results),
    "total_estimated_cost_usd": round(
        sum(r["estimated_cost_usd"] for r in individual_results), 6
    ),
}

output = {
    "generator_model": GENERATOR_MODEL,
    "judge_model": JUDGE_MODEL,
    "individual_results": individual_results,
    "aggregate": aggregate,
}

with open("evaluation_results.json", "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

aggregate

{'num_cases': 5,
 'average_score': 3.6,
 'min_score': 1,
 'max_score': 5,
 'total_time_seconds': 8.76,
 'total_input_tokens': 1903,
 'total_output_tokens': 509,
 'total_estimated_cost_usd': 0.000592}

## 5. Analyze and summarize results

In [7]:
print("=" * 60)
print("PER-CASE RESULTS")
print("=" * 60)

for r in individual_results:
    criteria = r["criteria_met"]
    num_met = sum(1 for v in criteria.values() if v is True)
    reasoning_short = (r["reasoning"] or "")[:120]
    print(f"\n[{r['id']}] expected: {r['expected_quality']}")
    print(f"  score: {r['score']}/5   criteria met: {num_met}/{len(criteria)} {criteria}")
    print(f"  time: {r['time_seconds']}s   tokens: {r['input_tokens']} in / {r['output_tokens']} out")
    print(f"  reasoning: {reasoning_short}...")

print("\n" + "=" * 60)
print("AGGREGATE SUMMARY")
print("=" * 60)
for k, v in aggregate.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 60)
print("CALIBRATION CHECK (fixed cases only, known expected quality)")
print("=" * 60)
quality_rank = {"bad": 1, "borderline": 3, "good": 5}
for r in individual_results:
    if r["expected_quality"] in quality_rank:
        expected_score = quality_rank[r["expected_quality"]]
        match = "OK" if abs(r["score"] - expected_score) <= 1 else "CHECK"
        print(f"  {r['id']}: expected ~{expected_score}, judge gave {r['score']}  [{match}]")

PER-CASE RESULTS

[live_1] expected: unknown (live generation, judge score is the real test)
  score: 5/5   criteria met: 3/3 {'acknowledgment': True, 'concrete_resolution': True, 'tone': True}
  time: 1.365s   tokens: 416 in / 96 out
  reasoning: The response effectively acknowledges the customer's frustration with the broken product and their negative experience. ...

[live_2] expected: unknown (live generation, judge score is the real test)
  score: 4/5   criteria met: 2/3 {'acknowledgment': True, 'concrete_resolution': True, 'tone': False}
  time: 2.489s   tokens: 398 in / 112 out
  reasoning: The response acknowledges the customer's frustration with the broken item and the repeated issues, which is a strong poi...

[fixed_good] expected: good
  score: 5/5   criteria met: 3/3 {'acknowledgment': True, 'concrete_resolution': True, 'tone': True}
  time: 1.669s   tokens: 381 in / 111 out
  reasoning: The response directly acknowledges the customer's specific complaint about the cracked